# Evaluation

Load one published stage, measure it on the train and validation splits, print
greedy samples, and score the reference chat tasks.

Evaluation is forward only. This notebook writes no checkpoint. It writes one
JSON report and pushes it beside the stage folder.


## Setup

The setup clones the repository, reads the Kaggle secrets, builds the shared
library for the T4 (`sm_75`), and checks the card count. See
`docs/checkpoint-format.md` section 15.


In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/ming6ao/nanochat.cpp"
REPO = Path("/kaggle/working/nanochat.cpp")
BASE = Path("/kaggle/working/nanochat-cache")

if not (REPO / ".git").is_dir():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
sys.path[:0] = [
    str(REPO / "python"),
    str(REPO / "notebooks"),
    str(REPO / ".agents" / "skills" / "model-fit" / "scripts"),
]


In [ ]:
import session

sess = session.setup(REPO_URL, REPO, arch="sm_75", devices=1, base=BASE)
print("library:", sess.library)
print("secrets:", sorted(sess.secrets))


In [ ]:
import nanochat_cpp as nc

for device in nc.devices():
    print(device, device.name,
          f"{device.total_memory_bytes / 2**30:.1f} GiB")


## Load the stage

`hub_checkpoint.load_stage` downloads the stage and builds the model, the
tokenizer, and the model arguments. The checkpoint load is parameter only, so
the reader skips the pretraining optimizer section.


In [ ]:
import hub_checkpoint

RUN_NAME = "base-L16-H1024-n8-kv4-s4096"
REPO_ID = hub_checkpoint.repo_id_for(RUN_NAME)
STAGE = "base"          # "base" or "sft"
REVISION = "base-500"   # the tag the pretraining notebook created

model, tokenizer, model_args = hub_checkpoint.load_stage(
    REPO_ID, STAGE, BASE / "hub", revision=REVISION, device="cuda")
config = model_args.to_config()
print(REPO_ID, STAGE, REVISION)
print("device:", model.device, "parameters:", f"{model.param_total():,}")


## Data

Download the held-out validation shard. The bits-per-byte loop reads the train
shard and the validation shard.


In [ ]:
import os
import subprocess
import sys

DATA = BASE / "base_data_climbmix"
TRAIN_SHARD = "shard_00000.parquet"
VAL_SHARD = "shard_06542.parquet"

if not (DATA / VAL_SHARD).is_file():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                    "huggingface_hub"], check=True)
    from huggingface_hub import snapshot_download

    snapshot_download(repo_id="karpathy/climbmix-400b-shuffle",
                      repo_type="dataset", local_dir=str(DATA),
                      allow_patterns=[TRAIN_SHARD, VAL_SHARD],
                      token=os.environ.get("HF_TOKEN"))


## Bits per byte

One `nc.Evaluator.bpb` call runs several steps. The cell splits the budget so
it can print progress. `EVAL_CHUNKS = 1` runs one atomic call.


In [ ]:
print("device:", model.device, "parameters:", f"{model.param_total():,}")

EVAL_TOKENS = 40 * 524288
EVAL_BATCH = 8
EVAL_CHUNKS = 8   # progress lines per split; 1 runs one atomic call


def base_bpb_for(parquet, split):
    data = nc.TokenData(parquet=str(parquet), tokenizer=tokenizer,
                        seq_len=config.seq_len, batch=EVAL_BATCH, seed=44)
    evaluator = nc.Evaluator(model)
    total = max(1, EVAL_TOKENS // (EVAL_BATCH * config.seq_len))
    chunks = max(1, min(EVAL_CHUNKS, total))
    per_chunk = total // chunks
    values = []
    for index in range(chunks):
        steps = per_chunk if index < chunks - 1 else total - per_chunk * index
        if steps <= 0:
            continue
        values.append(evaluator.bpb(data, steps))
        done = per_chunk * (index + 1)
        print(f"  {split}: {done}/{total} steps  "
              f"bpb ~{sum(values) / len(values):.4f}", flush=True)
    bpb = sum(values) / len(values) if values else float("nan")
    print(f"{STAGE} {split:>5} bpb: {bpb:.4f}", flush=True)
    return bpb


base_bpb = {"train": base_bpb_for(DATA / TRAIN_SHARD, "train"),
            "val": base_bpb_for(DATA / VAL_SHARD, "val")}
print(f"{STAGE} validation bits per byte: {base_bpb['val']:.4f}", flush=True)


## Samples

Greedy completions of a fixed prompt set.


In [ ]:
SAMPLE_PROMPTS = [
    "The capital of France is",
    "The chemical symbol of gold is",
    "If yesterday was Friday, then tomorrow will be",
    "The opposite of hot is",
    "The planets of the solar system are:",
    "My favorite color is",
    "If 5*x + 3 = 13, then x is",
]
bos = tokenizer.get_bos_token_id()
for prompt in SAMPLE_PROMPTS:
    prompt_ids = ([bos] if bos >= 0 else []) + tokenizer.encode(prompt)
    row = model.generate(prompt_ids, max_tokens=16, temperature=0.0)[0]
    generated = [token for token, keep in zip(row.tokens, row.mask) if keep]
    print("-" * 40)
    print(prompt + tokenizer.decode(generated))


## Chat tasks

`nc.chat.ChatEvaluator` scores the reference tasks with the loaded tokenizer.
A task that fails becomes a skipped report, so one missing dataset does not
stop the run.


In [ ]:
EVAL_TASKS = ["ARC-Easy", "ARC-Challenge", "MMLU", "GSM8K", "HumanEval"]
chat_report = nc.chat.ChatEvaluator(model, tokenizer=tokenizer).run(
    tasks=EVAL_TASKS, max_problems=20, seed=0)
for task in chat_report.tasks:
    status = "skipped" if task.skipped else f"{task.accuracy:.3f}"
    print(f"{task.name:>14}  {status:>8}  ({task.correct}/{task.total})")
print("categorical core:", chat_report.categorical_core)
print("chat core:", chat_report.chat_core)


## Report

Write one JSON report and push it beside the stage folder.


In [ ]:
import datetime
import json
import subprocess

git_revision = subprocess.run(
    ["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"],
    capture_output=True, text=True).stdout.strip()
report_data = {
    "stage": STAGE,
    "revision": REVISION,
    "repo_id": REPO_ID,
    "run_name": RUN_NAME,
    "git_revision": git_revision,
    "bits_per_byte": base_bpb,
    "chat": [
        {"task": task.name, "accuracy": task.accuracy,
         "correct": task.correct, "total": task.total,
         "skipped": task.skipped}
        for task in chat_report.tasks
    ],
    "created_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
}
report_name = f"reports/{STAGE}-{REVISION}.json"
report_path = BASE / "reports" / f"{STAGE}-{REVISION}.json"
report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text(json.dumps(report_data, indent=1, sort_keys=True) + "\n",
                       encoding="utf-8")
hub_checkpoint.push_file(REPO_ID, report_name, report_path,
                         token=os.environ.get("HF_TOKEN"))
print("report:", report_name)


## Next

Set `STAGE = "sft"` and `REVISION` to the `sft` tag to measure the fine-tuned
stage. The load path is the same.
